# Employee Attrition Prediction using Machine Learning

**IBM SkillsBuild | AICTE | BharatCares**  
**Data Analytics with AI Academic Internship 2026**

**Author:** Baskula Spandana  
**Date:** September 2026

---

### How to use this notebook
1. Upload the dataset file (`WA_Fn-UseC_-HR-Employee-Attrition.csv`) using the folder icon on the left
2. Run the cells one by one (or Runtime → Run all)


## 1. Install Required Libraries


In [ ]:
!pip install imbalanced-learn xgboost -q
print("✅ Libraries installed successfully!")


## 2. Import Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, f1_score,
                             precision_score, recall_score, accuracy_score)

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

sns.set_palette('husl')
%matplotlib inline
print("✅ All libraries imported successfully!")


## 3. Load the Dataset

Upload **WA_Fn-UseC_-HR-Employee-Attrition.csv** from:  
https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset


In [ ]:
df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()


## 4. Dataset Overview


In [ ]:
print("="*60)
print("DATASET INFO")
print("="*60)
df.info()
print("\nMissing Values:", df.isnull().sum().sum())
print("Duplicate Rows:", df.duplicated().sum())
print("\nAttrition Distribution:")
print(df['Attrition'].value_counts())
print(df['Attrition'].value_counts(normalize=True)*100)


## 5. Exploratory Data Analysis


In [ ]:
# Attrition distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(x='Attrition', data=df, ax=axes[0], palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Attrition Count', fontsize=14, fontweight='bold')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', (p.get_x()+p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=12)

df['Attrition'].value_counts().plot.pie(autopct='%1.1f%%', ax=axes[1],
                                         colors=['#2ecc71', '#e74c3c'],
                                         explode=(0, 0.05), shadow=True)
axes[1].set_title('Attrition Percentage', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()


In [ ]:
# Key categorical features vs Attrition
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(x='OverTime', hue='Attrition', data=df, ax=axes[0,0], palette=['#2ecc71','#e74c3c'])
axes[0,0].set_title('Attrition by OverTime', fontweight='bold')

sns.countplot(x='Department', hue='Attrition', data=df, ax=axes[0,1], palette=['#2ecc71','#e74c3c'])
axes[0,1].set_title('Attrition by Department', fontweight='bold')
axes[0,1].tick_params(axis='x', rotation=15)

sns.countplot(x='JobSatisfaction', hue='Attrition', data=df, ax=axes[1,0], palette=['#2ecc71','#e74c3c'])
axes[1,0].set_title('Attrition by Job Satisfaction', fontweight='bold')

sns.countplot(x='WorkLifeBalance', hue='Attrition', data=df, ax=axes[1,1], palette=['#2ecc71','#e74c3c'])
axes[1,1].set_title('Attrition by Work-Life Balance', fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Numerical features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(x='Attrition', y='MonthlyIncome', data=df, ax=axes[0], palette=['#2ecc71','#e74c3c'])
axes[0].set_title('Monthly Income vs Attrition', fontweight='bold')

sns.boxplot(x='Attrition', y='Age', data=df, ax=axes[1], palette=['#2ecc71','#e74c3c'])
axes[1].set_title('Age vs Attrition', fontweight='bold')

sns.boxplot(x='Attrition', y='YearsAtCompany', data=df, ax=axes[2], palette=['#2ecc71','#e74c3c'])
axes[2].set_title('Years at Company vs Attrition', fontweight='bold')

plt.tight_layout()
plt.show()


## 6. Data Preprocessing


In [ ]:
# Drop constant / ID columns
df_clean = df.drop(['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours'], axis=1)

# Encode target
df_clean['Attrition'] = df_clean['Attrition'].map({'Yes': 1, 'No': 0})

# Encode categorical columns
cat_cols = df_clean.select_dtypes(include='object').columns.tolist()
le = LabelEncoder()
for col in cat_cols:
    df_clean[col] = le.fit_transform(df_clean[col])

print("Categorical columns encoded:", cat_cols)
print("\nFinal shape:", df_clean.shape)
df_clean.head()


In [ ]:
X = df_clean.drop('Attrition', axis=1)
y = df_clean['Attrition']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain Attrition %:")
print(y_train.value_counts(normalize=True)*100)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# SMOTE
print("\nBefore SMOTE:", y_train.value_counts().to_dict())
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
print("After SMOTE:", pd.Series(y_train_res).value_counts().to_dict())


## 7. Model Building & Evaluation


In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print("="*60)
    print(f"MODEL: {model_name}")
    print("="*60)
    print(classification_report(y_test, y_pred, target_names=['No Attrition', 'Attrition']))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    acc = accuracy_score(y_test, y_pred)

    print(f"\nAccuracy  : {acc:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    print(f"ROC-AUC   : {roc_auc:.4f}")

    return {'Model': model_name, 'Accuracy': acc, 'Precision': precision,
            'Recall': recall, 'F1-Score': f1, 'ROC-AUC': roc_auc,
            'y_pred': y_pred, 'y_prob': y_prob, 'cm': confusion_matrix(y_test, y_pred)}


### 7.1 Logistic Regression


In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_res, y_train_res)
lr_results = evaluate_model(lr_model, X_test_scaled, y_test, "Logistic Regression")


### 7.2 Random Forest


In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train_res, y_train_res)
rf_results = evaluate_model(rf_model, X_test_scaled, y_test, "Random Forest")


### 7.3 XGBoost


In [ ]:
xgb_model = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                          random_state=42, eval_metric='logloss', n_jobs=-1)
xgb_model.fit(X_train_res, y_train_res)
xgb_results = evaluate_model(xgb_model, X_test_scaled, y_test, "XGBoost")


## 8. Model Comparison


In [ ]:
results_df = pd.DataFrame([
    {k:v for k,v in lr_results.items() if k not in ['y_pred','y_prob','cm']},
    {k:v for k,v in rf_results.items() if k not in ['y_pred','y_prob','cm']},
    {k:v for k,v in xgb_results.items() if k not in ['y_pred','y_prob','cm']}
])
print("MODEL PERFORMANCE COMPARISON")
print("="*60)
print(results_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
metrics = ['Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metrics))
width = 0.25
axes[0].bar(x-width, results_df.iloc[0][metrics], width, label='Logistic Regression', color='#3498db')
axes[0].bar(x, results_df.iloc[1][metrics], width, label='Random Forest', color='#2ecc71')
axes[0].bar(x+width, results_df.iloc[2][metrics], width, label='XGBoost', color='#e74c3c')
axes[0].set_ylabel('Score')
axes[0].set_title('Model Performance Comparison', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].set_ylim(0, 1.1)

for res, color, name in [(lr_results,'#3498db','Logistic Regression'),
                         (rf_results,'#2ecc71','Random Forest'),
                         (xgb_results,'#e74c3c','XGBoost')]:
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    axes[1].plot(fpr, tpr, color=color, label=f"{name} (AUC={res['ROC-AUC']:.3f})")
axes[1].plot([0,1],[0,1],'k--')
axes[1].set_title('ROC Curves', fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, res, title in zip(axes, [lr_results, rf_results, xgb_results],
                          ['Logistic Regression', 'Random Forest', 'XGBoost']):
    sns.heatmap(res['cm'], annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No','Yes'], yticklabels=['No','Yes'])
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
plt.tight_layout()
plt.show()


## 9. Feature Importance


In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance.head(15), palette='viridis')
plt.title('Top 15 Features Driving Employee Attrition (XGBoost)', fontweight='bold')
plt.tight_layout()
plt.show()
print(feature_importance.head(10).to_string(index=False))


## 10. Key Insights & Recommendations

### Key Findings
1. Attrition rate is approximately **16%**.
2. Employees who work **OverTime** show significantly higher attrition.
3. Lower **Job Satisfaction**, **Work-Life Balance**, and **Monthly Income** are strong indicators of leaving.
4. Younger employees and those with fewer years at the company tend to leave more.
5. Tree-based models (Random Forest / XGBoost) generally perform best.

### Business Recommendations
- Reduce excessive overtime through better workload distribution.
- Improve job satisfaction and work-life balance programs.
- Focus retention efforts on high-risk groups (young employees, low satisfaction, frequent overtime).
- Use the model to flag employees at risk so HR can take proactive action.

---
**Thank you!**  
**Baskula Spandana**  
IBM SkillsBuild Data Analytics with AI Internship 2026  
BharatCares × AICTE × IBM
